# Scenario 03 — Source Update

A client publishes a source, then updates its `minPayment` on-chain via `Source.set_params()`
(which goes through `Nexus.updateSource` so the `SourceUpdated` event fires).

**Role:** Client only.

**Prerequisites:** `CLIENT_PRIVATE_KEY`.

**Flow:**

```
Client
  │
  ├─ 1. publish_source       → Source(minPayment=X)
  ├─ 2. source.set_params    → Receipt
  └─ 3. source.get_params()  → minPayment == Y
```


## 0. Setup


In [ ]:
import os
import time

from web3 import Web3

from ogpu import ChainConfig, ChainId
from ogpu.client import (
    DeliveryMethod,
    ImageEnvironments,
    SourceInfo,
    TaskInfo,
    TaskInput,
    publish_source,
    publish_task,
)
from ogpu.protocol import Master, Provider, Response, Source, Task, vault

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

CLIENT_KEY = os.environ["CLIENT_PRIVATE_KEY"]


## 1. Publish with a starting minPayment


In [ ]:
initial = SourceInfo(
    name="update-demo",
    description="scenario 03",
    logoUrl="https://example.com/logo.png",
    imageEnvs=ImageEnvironments(
        cpu="https://cipfs.ogpuscan.io/ipfs/QmNWFLL13ujf3KUTJvfNx42bA5fWDV96qqUdjY6nwpuwD9",
    ),
    minPayment=Web3.to_wei(0.01, "ether"),
    minAvailableLockup=0,
    maxExpiryDuration=86400,
    deliveryMethod=DeliveryMethod.FIRST_RESPONSE,
)

source = publish_source(initial, private_key=CLIENT_KEY)
print(f"Source: {source.address}")
print(f"Initial minPayment: {source.get_params().minPayment}")


## 2. Update params

`set_params` takes a full `SourceParams` tuple — the easy way is to build a new `SourceInfo`
and convert it via `to_source_params`. IPFS metadata gets re-published in the process.


In [ ]:
from eth_account import Account

updated_info = SourceInfo(
    name="update-demo",
    description="scenario 03 — updated",
    logoUrl="https://example.com/logo.png",
    imageEnvs=ImageEnvironments(
        cpu="https://cipfs.ogpuscan.io/ipfs/QmNWFLL13ujf3KUTJvfNx42bA5fWDV96qqUdjY6nwpuwD9",
    ),
    minPayment=Web3.to_wei(0.05, "ether"),  # bumped
    minAvailableLockup=0,
    maxExpiryDuration=86400,
    deliveryMethod=DeliveryMethod.FIRST_RESPONSE,
)

client_addr = Account.from_key(CLIENT_KEY).address
new_params = updated_info.to_source_params(client_addr)

receipt = source.set_params(new_params, signer=CLIENT_KEY)
print(f"Update tx: {receipt.tx_hash}")


## 3. Verify the on-chain value changed


In [ ]:
p = source.get_params()
print(f"New minPayment: {p.minPayment}")
assert p.minPayment == Web3.to_wei(0.05, "ether")
